In [ ]:
# Author: Sachuriga
# Email: sachuriga3@gmail.com
# Purpose: Process CSV files and add their columns to corresponding NWB files

import pandas as pd
from pynwb import NWBHDF5IO
import os

target_prefixes_control= ['65165', '65091', '63383', '66539','65622','283']
target_prefixes_exp = ['65588', '63385', '66538', '66537','66922']


# Set your base folder path
base_folder = 'S:/Sachuriga/nwb/test4neo'

# Get all files in the directory
all_files = os.listdir(base_folder)

# Filter for CSV and NWB files and create pairs
csv_files = [f for f in all_files if f.endswith('.csv')]
nwb_files = [f for f in all_files if f.endswith('.nwb')]

# Process each pair of files
for csv_file in csv_files:
    # Try to find a matching NWB file (assuming similar base names)
    base_name = os.path.splitext(csv_file)[0]
    nwb_file = f"{base_name}.nwb"
    
    if nwb_file in nwb_files:
        csvpath = os.path.join(base_folder, csv_file)
        nwbpath = os.path.join(base_folder, nwb_file)
        
        try:
            # Read the CSV file
            df = pd.read_csv(csvpath)
            
            # Get column names
            column_names = df.columns.tolist()
            
            # Open and modify NWB file
            with NWBHDF5IO(nwbpath, "a") as io:
                nwb = io.read()
                for col in column_names:
                    print(f"Processing column: {col}")
                    nwb.units.add_column(
                        name=f"matlab_{col}",
                        description=f"values of {col}",
                        data=df[col].tolist()
                    )
        
                # Uncomment these if you need to modify subject info
                # nwb.subject.genotype = "NDNF-flp +/- and Pde1c +/-"
                # nwb.subject.set_modified()


                # nwb.subject.genotype = "NDNF-flp-/- and Pde1c -/-"
                
                io.write(nwb)
            print(f"Successfully processed {csv_file} into {nwb_file}")
            
        except Exception as e:
            print(f"Error processing {csv_file}: {str(e)}")
    else:
        print(f"Warning: No matching NWB file found for {csv_file}")

print("Processing complete!")

In [ ]:
import pandas as pd
from pynwb import NWBHDF5IO
import os
from pathlib import Path
from pynwb.file import Subject

# Output path for the summary table
output_path = r'Q:/sachuriga/Temporal/CR_CA1_paper/tables/'

# Set your base folder path
base_folder = 'S:/Sachuriga/nwb/test4neo/'

# Define animal groups
target_prefixes_control = ['65165', '65091', '63383', '66539', '65622']
target_prefixes_exp = ['65588', '63385', '66538', '66537', '66922']
males = ['65165', '63383', '66539', '63385', '65622']
females = ['65091', '65588', '66538', '66537', '66922']
surgry_sachuriga = ['65165', '63383', '66539', '63385', '65091', '65588', '66538', '66537', '65283']
surgry_ingvild = ['66922', '65622']

# Initialize a dictionary to store summary data for each animal
summary_data = {}

# Get all NWB files in the directory
all_files = os.listdir(base_folder)
nwb_files = [f for f in all_files if f.endswith('.nwb')]
print(f"Found {len(nwb_files)} NWB files: {nwb_files}")

# Process each NWB file
for nwb_file in nwb_files:
    nwbpath = os.path.join(base_folder, nwb_file)
    print(f"Processing {nwbpath}")

    # Open and modify NWB file for metadata
    with NWBHDF5IO(Path(nwbpath), "a") as io:
        nwb = io.read()
        animal_id = nwbpath.split('/')[-1][:5]
        print(f"Animal ID: {animal_id}")

        # Assign sex
        sex = "M" if animal_id in males else "F" if animal_id in females else "Unknown"

        # Initialize genotype and experimenter as None in case they aren't set
        genotype = None
        experimenter = None

        # Create or update subject
        try:
            subject = Subject(
                subject_id=animal_id,
                age="P90D",
                description="mouse 5",
                species="Mus musculus",
                sex=sex)
            nwb.subject = subject
            
            if animal_id in target_prefixes_control:
                nwb.subject.genotype = "NDNF-flp-/- and Pde1c -/-"
            elif animal_id in target_prefixes_exp:
                nwb.subject.genotype = "NDNF-flp +/- and Pde1c +/-"

            if animal_id in surgry_sachuriga:
                nwb.experimenter = ("Sachuriga",)  # NWB expects a tuple for experimenter
            elif animal_id in surgry_ingvild:
                nwb.experimenter = ("Ingvild Lynneberg Glærum",)

            # Capture genotype and experimenter after setting them
            genotype = nwb.subject.genotype
            experimenter = nwb.experimenter[0] if nwb.experimenter else "Unknown"

        except AttributeError:
            print(f"AttributeError for {animal_id}, skipping subject update")
            genotype = nwb.subject.genotype
            experimenter = nwb.experimenter[0] if nwb.experimenter else "Unknown"

        io.write(nwb)

    # Open the NWB file again to process units
    with NWBHDF5IO(Path(nwbpath), "a") as io:
        nwb = io.read()
        unit_table = nwb.units.to_dataframe()

        # Initialize counters if animal_id not in summary_data
        if animal_id not in summary_data:
            summary_data[animal_id] = {
                "Animal ID": animal_id,
                "Sex": sex,
                "Genotype": genotype,  # Add genotype to summary
                "Experimenter": experimenter,  # Add experimenter to summary
                "Sessions": 0,
                "Good Units": 0,
                "Pyramidal Cells": 0,
                "Narrow Spike Interneurons": 0,
                "Wide Spike Interneurons": 0,
                "Speed Cells": 0,
                "Place Cells": 0
            }

        # Increment session count
        # Assuming sessions are counted as the number of NWB files per animal (since we're iterating over files)
        # If sessions are stored in nwb.intervals['epochs'], uncomment the line below and adjust accordingly
        summary_data[animal_id]["Sessions"] += 1
        # total_sessions = len(nwb.intervals['epochs']) if 'epochs' in nwb.intervals else 0
        # summary_data[animal_id]["Sessions"] += total_sessions

        # Lists to store unit classifications
        unit_quality = []
        cell_type = []
        functional_cell_type = []

        # Classify each unit
        for i in range(len(unit_table.index)):
            # Identify good units
            if (unit_table['isi_violations_ratio'][i] <= 0.02 and 
                unit_table['l_ratio'][i] <= 0.05 and 
                unit_table['amplitude_median'][i] <= -40):
                unit_quality.append("good")
                summary_data[animal_id]["Good Units"] += 1

                # Classify cell type
                if unit_table['peak_to_valley'][i] <= 0.000425:
                    cell_type.append("Narrow spike interneuron")
                    summary_data[animal_id]["Narrow Spike Interneurons"] += 1
                    if unit_table['matlab_speedScores'][i] >= 0.3:
                        functional_cell_type.append("Speed cell")
                        summary_data[animal_id]["Speed Cells"] += 1
                    else:
                        functional_cell_type.append("None")
                elif unit_table['peak_to_valley'][i] > 0.000425 and unit_table['matlab_acg_tau_rise'][i] >= 6:
                    cell_type.append("Wide Spike Interneurons")
                    summary_data[animal_id]["Wide Spike Interneurons"] += 1
                    if unit_table['matlab_speedScores'][i] >= 0.3:
                        functional_cell_type.append("Speed cell")
                        summary_data[animal_id]["Speed Cells"] += 1
                    else:
                        functional_cell_type.append("None")
                elif unit_table['peak_to_valley'][i] > 0.000425 and unit_table['matlab_acg_tau_rise'][i] < 6:
                    cell_type.append("Pyramidal cells")
                    summary_data[animal_id]["Pyramidal Cells"] += 1
                    if (unit_table['matlab_reject_h0'][i] == 1 and 
                        unit_table['matlab_test_stat_si'][i] >= 1.71 and 
                        unit_table['matlab_maxfsize'][i] > 20):
                        functional_cell_type.append("Place cell")
                        summary_data[animal_id]["Place Cells"] += 1
                    else:
                        functional_cell_type.append("None")
                else:
                    cell_type.append("None")
                    functional_cell_type.append("None")
            else:
                unit_quality.append("bad")
                cell_type.append("None")
                functional_cell_type.append("None")

        # Add columns to the NWB file
        if "unit_quality" not in nwb.units.colnames:
            nwb.units.add_column(name="unit_quality", description="values of unit_quality", data=unit_quality)
        if "Cell Type" not in nwb.units.colnames:
            nwb.units.add_column(name="Cell Type", description="values of cell_type", data=cell_type)
        if "Functional Cell Type" not in nwb.units.colnames:
            nwb.units.add_column(name="Functional Cell Type", description="values of functional_cell_type", data=functional_cell_type)

        io.write(nwb)

# Create a DataFrame from the summary data
summary_df = pd.DataFrame(list(summary_data.values()))

# Reorder columns to match the desired output, including Genotype and Experimenter
summary_df = summary_df[[
    "Animal ID", "Sex", "Genotype", "Experimenter", "Sessions", "Good Units", 
    "Pyramidal Cells", "Narrow Spike Interneurons", "Wide Spike Interneurons", 
    "Speed Cells", "Place Cells"
]]

# Save the summary table to a CSV file
output_file = os.path.join(output_path, "animal_summary_table.csv")
summary_df.to_csv(output_file, index=False)
print(f"Summary table saved to {output_file}")

# Print the summary table
print(summary_df)

print("Processing complete!")

In [ ]:
import pandas as pd
from pynwb import NWBHDF5IO
import os
from pathlib import Path
from pynwb.file import Subject

# Output path for the summary table
output_path = r'Q:/sachuriga/Temporal/CR_CA1_paper/tables/'

# Set your base folder path
base_folder = 'S:/Sachuriga/nwb/test4neo/'

# Define animal groups
target_prefixes_control = ['65165', '65091', '63383', '66539', '65622']
target_prefixes_exp = ['65588', '63385', '66538', '66537', '66922']
males = ['65165', '63383', '66539', '63385', '65622']
females = ['65091', '65588', '66538', '66537', '66922']
surgry_sachuriga = ['65165', '63383', '66539', '63385', '65091', '65588', '66538', '66537', '65283']
surgry_ingvild = ['66922', '65622']

# Initialize a dictionary to store summary data for each animal
summary_data = {}

# Get all NWB files in the directory
all_files = os.listdir(base_folder)
nwb_files = [f for f in all_files if f.endswith('.nwb')]
print(f"Found {len(nwb_files)} NWB files: {nwb_files}")

# Process each NWB file
for nwb_file in nwb_files:
    nwbpath = os.path.join(base_folder, nwb_file)
    print(f"Processing {nwbpath}")

    # Open and modify NWB file for metadata
    with NWBHDF5IO(Path(nwbpath), "a") as io:
        nwb = io.read()
        animal_id = nwbpath.split('/')[-1][:5]
        print(f"Animal ID: {animal_id}")

        # Assign sex
        sex = "M" if animal_id in males else "F" if animal_id in females else "Unknown"

        # Initialize genotype and experimenter as None in case they aren't set
        genotype = None
        experimenter = None

        # Create or update subject
        try:
            subject = Subject(
                subject_id=animal_id,
                age="P90D",
                description="mouse 5",
                species="Mus musculus",
                sex=sex)
            nwb.subject = subject
            
            if animal_id in target_prefixes_control:
                nwb.subject.genotype = "NDNF-flp-/- and Pde1c -/-"
            elif animal_id in target_prefixes_exp:
                nwb.subject.genotype = "NDNF-flp +/- and Pde1c +/-"

            if animal_id in surgry_sachuriga:
                nwb.experimenter = ("Sachuriga",)  # NWB expects a tuple for experimenter
            elif animal_id in surgry_ingvild:
                nwb.experimenter = ("Ingvild Lynneberg Glærum",)

            # Capture genotype and experimenter after setting them
            genotype = nwb.subject.genotype
            experimenter = nwb.experimenter[0] if nwb.experimenter else "Unknown"

        except AttributeError:
            print(f"AttributeError for {animal_id}, skipping subject update")
            genotype = nwb.subject.genotype
            experimenter = nwb.experimenter[0] if nwb.experimenter else "Unknown"

        io.write(nwb)

    # Open the NWB file again to process units
    with NWBHDF5IO(Path(nwbpath), "a") as io:
        nwb = io.read()
        unit_table = nwb.units.to_dataframe()

        # Initialize counters if animal_id not in summary_data
        if animal_id not in summary_data:
            summary_data[animal_id] = {
                "Animal ID": animal_id,
                "Sex": sex,
                "Genotype": genotype,  # Add genotype to summary
                "Experimenter": experimenter,  # Add experimenter to summary
                "Sessions": 0,
                "Total Units": 0,  # New field for total units
                "Good Units": 0,
                "Pyramidal Cells": 0,
                "Narrow Spike Interneurons": 0,
                "Wide Spike Interneurons": 0,
                "Speed Cells": 0,
                "Place Cells": 0
            }

        # Increment session count
        summary_data[animal_id]["Sessions"] += 1

        # Count total units
        total_units = len(unit_table.index)
        summary_data[animal_id]["Total Units"] += total_units  # Add total units to summary

        # Lists to store unit classifications
        unit_quality = []
        cell_type = []
        functional_cell_type = []

        # Classify each unit
        for i in range(len(unit_table.index)):
            # Count every unit regardless of quality (already handled above with total_units)
            
            # Identify good units
            if (unit_table['isi_violations_ratio'][i] <= 0.2 and 
                unit_table['l_ratio'][i] <= 0.05 and 
                unit_table['amplitude_median'][i] <= -40):
                unit_quality.append("good")
                summary_data[animal_id]["Good Units"] += 1

                # Classify cell type
                if unit_table['peak_to_valley'][i] <= 0.000425:
                    cell_type.append("Narrow spike interneuron")
                    summary_data[animal_id]["Narrow Spike Interneurons"] += 1
                    if unit_table['matlab_speedScores'][i] >= 0.3:
                        functional_cell_type.append("Speed cell")
                        summary_data[animal_id]["Speed Cells"] += 1
                    else:
                        functional_cell_type.append("None")
                elif unit_table['peak_to_valley'][i] > 0.000425 and unit_table['matlab_acg_tau_rise'][i] >= 6:
                    cell_type.append("Wide Spike Interneurons")
                    summary_data[animal_id]["Wide Spike Interneurons"] += 1
                    if unit_table['matlab_speedScores'][i] >= 0.3:
                        functional_cell_type.append("Speed cell")
                        summary_data[animal_id]["Speed Cells"] += 1
                    else:
                        functional_cell_type.append("None")
                elif unit_table['peak_to_valley'][i] > 0.000425 and unit_table['matlab_acg_tau_rise'][i] < 6:
                    cell_type.append("Pyramidal cells")
                    summary_data[animal_id]["Pyramidal Cells"] += 1
                    if (unit_table['matlab_reject_h0'][i] == 1 and 
                        unit_table['matlab_test_stat_si'][i] >= 1.72 and 
                        unit_table['matlab_maxfsize'][i] > 20):
                        functional_cell_type.append("Place cell")
                        summary_data[animal_id]["Place Cells"] += 1
                    else:
                        functional_cell_type.append("None")
                else:
                    cell_type.append("None")
                    functional_cell_type.append("None")
            else:
                unit_quality.append("bad")
                cell_type.append("None")
                functional_cell_type.append("None")

        # Add columns to the NWB file
        if "unit_quality" not in nwb.units.colnames:
            nwb.units.add_column(name="unit_quality", description="values of unit_quality", data=unit_quality)
        if "Cell Type" not in nwb.units.colnames:
            nwb.units.add_column(name="Cell Type", description="values of cell_type", data=cell_type)
        if "Functional Cell Type" not in nwb.units.colnames:
            nwb.units.add_column(name="Functional Cell Type", description="values of functional_cell_type", data=functional_cell_type)

        io.write(nwb)

# Create a DataFrame from the summary data
summary_df = pd.DataFrame(list(summary_data.values()))

# Reorder columns to match the desired output, including Total Units, Genotype, and Experimenter
summary_df = summary_df[[
    "Animal ID", "Sex", "Genotype", "Experimenter", "Sessions", "Total Units", "Good Units", 
    "Pyramidal Cells", "Narrow Spike Interneurons", "Wide Spike Interneurons", 
    "Speed Cells", "Place Cells"
]]

# Save the summary table to an Excel file
output_file = os.path.join(output_path, "animal_summary_table.xlsx")
summary_df.to_excel(output_file, index=False)
print(f"Summary table saved to {output_file}")

# Print the summary table
print(summary_df)

print("Processing complete!")

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Example data - replace with your actual data
data = np.random.normal(0, 1, (1000, 2))  # Example data with two features

# Clear previous figure
plt.clf()

# Create jointplot with KDE using a blue color scheme
g = sns.jointplot(x=data[:, 0], y=data[:, 1], kind='kde', 
                 cmap='Blues', shade=True, levels=6, 
                 color='blue', fill=True)

# Add cumulative contours (optional, to approximate CDF-like behavior)
g.plot_joint(sns.kdeplot, cumulative=True, levels=5, colors='blue', alpha=0.5,linewidths=0)

# Customize marginal plots (to match the example)
g.ax_marg_x.hist(data[:, 0], bins=30, color='blue', alpha=0.5)
g.ax_marg_y.hist(data[:, 1], bins=30, color='blue', alpha=0.5, orientation='horizontal')

# Add Pearson correlation (example values, replace with your data's correlation)
pearson_corr = np.corrcoef(data[:, 0], data[:, 1])[0, 1]
p_value = 0.15  # Replace with your actual p-value
g.ax_joint.text(0.05, 0.95, f'pearsonr = {pearson_corr:.2f}, p = {p_value:.2f}', 
                transform=g.ax_joint.transAxes, 
                bbox=dict(facecolor='white', alpha=0.8))

# Customize the plot
g.fig.suptitle('Density of Features', y=1.02)
g.fig.tight_layout()

# Save the figure

# Show the plot
plt.show()

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Example data - replace with your actual data
data = np.random.normal(0, 1, (1000, 2))

# Clear previous figure
plt.clf()

# Create jointplot with KDE, removing contour lines
g = sns.jointplot(x=data[:, 0], y=data[:, 1], kind='kde', 
                 cmap='light:b', shade=True, fill=True, 
                 joint_kws={'contour': False})

# Add Pearson correlation (example values)
pearson_corr = np.corrcoef(data[:, 0], data[:, 1])[0, 1]
p_value = 0.15
g.ax_joint.text(0.05, 0.95, f'pearsonr = {pearson_corr:.2f}, p = {p_value:.2f}', 
                transform=g.ax_joint.transAxes, 
                bbox=dict(facecolor='white', alpha=0.8))

# Customize the plot
g.fig.suptitle('Density of Features', y=1.02)
g.fig.tight_layout()

# Save the figure
#g.savefig('graphs/density.svg')

# Show the plot
plt.show()

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Example data - replace with your actual data
data = np.random.normal(0, 1, (1000, 2))

# Clear previous figure
plt.clf()

# Calculate 2D histogram and CDF
H, xedges, yedges = np.histogram2d(data[:, 0], data[:, 1], bins=50)
Hcdf = np.cumsum(np.cumsum(H, axis=0), axis=1)
Hcdf = Hcdf / Hcdf.max()  # Normalize to [0,1]

# Create figure
plt.figure(figsize=(8, 6))
plt.imshow(Hcdf.T, origin='lower', cmap='Blues', 
          extent=[xedges[0], xedges[-1], yedges[0], yedges[-1]])

# Optionally add KDE contours (without lines if you prefer)
sns.kdeplot(x=data[:, 0], y=data[:, 1], cumulative=True, 
           levels=5, cmap='Blues', fill=True, alpha=0.5, 
           linewidths=0)  # Set linewidths=0 to remove contour lines

# Add Pearson correlation (example values)
pearson_corr = np.corrcoef(data[:, 0], data[:, 1])[0, 1]
p_value = 0.15
plt.text(0.05, 0.95, f'pearsonr = {pearson_corr:.2f}, p = {p_value:.2f}', 
         transform=plt.gca().transAxes, 
         bbox=dict(facecolor='white', alpha=0.8))

# Customize
plt.colorbar(label='Cumulative Probability')
plt.suptitle('Density of Features', y=1.02)
plt.tight_layout()

# Save
plt.savefig('graphs/density.svg')
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter

# Read the CSV file
csv_path = r'S:/Sachuriga/nwb/test4neo/test/speed_analysis_results.csv'  # Adjust path if needed
df = pd.read_csv(csv_path, dtype={'H_counts_smoothed': float, 'speed_heatmap_smoothed': float})

# Define control and experimental animal IDs
# control_ids = ['65165', '65091', '63383', '66539', '65622']
# exp_ids = ['65588', '63385', '66538', '66537', '66922']


control_ids = [65165, 65091, 63383, 66539, 65622]
exp_ids = [65588,63385, 66538, 66537, 66922]

# Filter for 'A' sessions
df_a = df[df['session'] == 'A']

# Function to parse string array into numpy array
def parse_array(array_str):
    # Remove the surrounding quotes and split into rows
    array_str = array_str.strip('"')
    rows = array_str.split('\n')
    # Parse each row into numbers
    data = [list(map(float, row.split())) for row in rows if row.strip()]
    return np.array(data)

# Separate into control and experimental groups
control_df = df_a[df_a['animal_id'].isin(control_ids)]
exp_df = df_a[df_a['animal_id'].isin(exp_ids)]
exp_df
